# CO3133 A2 - Food-101 training (Kaggle, **GPU T4** notebook)

1. *Settings → Accelerator: GPU T4 x1* (P100 also works), *Internet: On*.
2. *Add input → Your Work →* the data-preparation notebook (folder `food-101-256`).
3. Optional, to **resume** runs from an earlier session: add that earlier training notebook's output
   as input too; its `outputs/runs` folder is copied back below and unfinished runs continue from `last.pt`.
4. Edit `RUNS` / `SEEDS`, then *Save Version → Save & Run All (Commit)* (runs unattended up to 12 h).
5. Download `results_bundle_*.zip` (and `checkpoints_*.zip`) from the output tab and unzip at the repo
   root, then `python scripts/compare.py`.

In [ ]:
import os, subprocess, sys
REPO_URL = "https://github.com/pbt245/CO3113-A2.git"
REF = "main"            # set a commit hash or tag for the final runs (traceability)
REPO = "/kaggle/working/CO3113-A2"
if not os.path.exists(REPO):
    subprocess.run(["git", "clone", REPO_URL, REPO], check=True)
subprocess.run(["git", "-C", REPO, "checkout", REF], check=True)
os.chdir(REPO)
print(subprocess.run(["git", "log", "-1", "--oneline"], capture_output=True, text=True).stdout)

In [ ]:
!pip install -q -r requirements.txt
!nvidia-smi

In [ ]:
import glob
def find_food101(pattern_root="/kaggle/input"):
    """Return every folder under /kaggle/input that has meta/classes.txt and images/."""
    hits = [os.path.dirname(os.path.dirname(p)) for p in glob.glob(f"{pattern_root}/**/meta/classes.txt", recursive=True)]
    return [h for h in hits if os.path.isdir(os.path.join(h, "images"))]
print(find_food101())

In [ ]:
roots = [r for r in find_food101() if "256" in r] or find_food101()
IMAGE_ROOT = roots[0]
print("images:", IMAGE_ROOT)

# resume: copy outputs/runs of an earlier session (if added as input)
import shutil
for prev in glob.glob("/kaggle/input/**/outputs/runs/food101", recursive=True):
    print("restoring", prev)
    shutil.copytree(prev, "outputs/runs/food101", dirs_exist_ok=True)

In [ ]:
# what to run in this session. M2: B0 + P1 (seed 42 first). See configs/food101/ for all IDs.
RUNS = ["b0_simple_cnn", "p1_resnet50"]
SEEDS = [42]
MAX_HOURS = 11.0      # per run; a run that hits it stops cleanly and resumes next session
COMMON = [f"data.image_root={IMAGE_ROOT}", "data.num_workers=4", "train.show_progress=false",
          f"train.max_hours={MAX_HOURS}"]

In [ ]:
# 1-epoch timing runs (300 steps) -> measured img/s replaces the proposal's assumptions
for exp, cfg in [("b0_simple_cnn", "configs/food101/b0_simple_cnn.yaml"),
                 ("p1_resnet50", "configs/food101/p1_resnet50.yaml")]:
    if exp in RUNS:
        !python scripts/train.py --config {cfg} --no-eval --overwrite --set {" ".join(COMMON)} train.epochs=1 train.max_steps_per_epoch=300 experiment.name=timing_{exp} experiment.group=debug

In [ ]:
!python scripts/run_experiments.py --models {" ".join(RUNS)} --seeds {" ".join(map(str, SEEDS))} --set {" ".join(COMMON)}

In [ ]:
!python scripts/compare.py
!python scripts/pack_results.py --out-dir /kaggle/working

In [ ]:
# best checkpoints for download / the checkpoint link (last.pt stays in outputs/runs for resuming)
!cd {REPO} && zip -q -r /kaggle/working/checkpoints_$(date +%Y%m%d-%H%M).zip outputs/runs/food101/*/seed*/checkpoints/best.pt
!ls -lh /kaggle/working